# Jupyter notebooks — quick background

A **notebook** is a document made of **cells**. Two main types:

| Type | What it is |
|------|------------|
| **Markdown** | Text, headings, bullet lists, math — what you are reading now. |
| **Code** | Python code. Run one cell at a time with **Shift+Enter** (runs and moves down) or the Run button. |

**Kernel** = the Python process that runs your code. Pick the same environment where you `pip install` packages (e.g. your project venv).

**Order matters**: variables from cell 3 exist after you run cell 3; run cells **top to bottom** the first time.

**Saving**: the `.ipynb` file stores cells + outputs. Commit it to git if you want to share results (large outputs can be cleared: *Clear All Outputs*).

**Notebooks vs `.py` scripts**: notebooks are great for **exploration, plots, and notes**. Move stable logic into `.py` modules when it grows.

# SerDes modeling — start here (Python only)

This notebook uses **`serdes_building_blocks/`** inside **SerdesProjectPython** (same API as the older `Serdes_v1_ffe_channel_ctle` tutorial: `serdes_channel`, `pam4_generator`).

- `SerdesProjectPython/path_setup.py` — standard way to add `serdes_building_blocks` to `sys.path`
- **Optional:** the workspace parent (e.g. `CursorProjPython/`) may still contain `Serdes_v1_ffe_channel_ctle/` for extra demos; use `add_serdes_v1_ffe_channel_ctle()` if you need that path without colliding with `serdes_channel` from building blocks (only one at a time in a kernel session)

The cells below use **only** `serdes_building_blocks` for the minimal example.

In [ ]:
from pathlib import Path
import sys

_sp = Path.cwd().resolve()
if _sp.name == "notebooks":
    _sp = _sp.parent
sys.path.insert(0, str(_sp))

from path_setup import serdes_project_root, add_building_blocks

add_building_blocks()
print("SerdesProjectPython:", serdes_project_root())
print("Using imports from: serdes_building_blocks/")

In [ ]:
# Plots appear inline in the notebook (not a separate window)
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from serdes_channel import SerdesChannel
from pam4_generator import PAM4Generator

## Minimal experiment (same idea as `simple_example.py`)

Create channel → compute response → random PAM4 → pass through channel → plot frequency response.

Tweak `length` (meters) or `symbol_rate` and re-run the cell.

In [ ]:
symbol_rate = 32e9
samples_per_symbol = 32
length_m = 0.1  # 10 cm

channel = SerdesChannel(
    symbol_rate=symbol_rate,
    samples_per_symbol=samples_per_symbol,
    length=length_m,
)
channel.compute_channel()

pam4 = PAM4Generator()
symbols = pam4.generate_random_symbols(1000)
signal_in = pam4.oversample(symbols, channel.samples_per_symbol)
signal_out = channel.apply_channel(signal_in)

channel.plot_frequency_response()
plt.show()

## Next steps (still Python-only)

1. Explore modules in `SerdesProjectPython/serdes_building_blocks/` (`serdes_system.py`, `joint_ffe_ml_equalizer.py`, …).
2. For notebooks that need legacy scripts only under the workspace parent, use `path_setup.workspace_root()` and run or `%run` files there — avoid putting two different `serdes_channel.py` trees on `sys.path` at once.
3. For Verilog verification notebooks, open **`getting_started.ipynb`** in this folder (it wires `Verilog_testing_v2` via `add_verilog_testing_v2()`).